# `llm_complete` MCP Buyer Notebook — the x402-over-MCP workflow for the chat tool (Deno/TypeScript)

Drives the **real, locally-running** `mcp_server.ts` (`cd scw_js && npm run dev:mcp`, port 8086)
as the official client stack would: the MCP SDK's `StreamableHTTPClientTransport` against our
hand-rolled stateless dispatcher, wrapped by `@x402/mcp`'s `createx402MCPClient` for payment —
no custom protocol code anywhere.

Focused on `llm_complete`, the only tool payable on testnet: the MCP endpoint is a **pure
translation layer** over the real handlers, so it inherits their cost rules — `web_search`
is mainnet-only by design (testnet coins must never buy metered Brave queries) and is left
to a mainnet run. `llm_complete` offers testnet and the handler **mocks completions there**,
so the whole paid walk below costs no real Mistral spend — only the channel deposits.

The spike questions this notebook answers:

1. Does the **official MCP client** tolerate our stateless-JSON server (405 on GET/DELETE, no sessions)?
2. Is our payment-required result the **x402-over-MCP convention** — does the official wrapper
   extract the demands and pay without custom code?
3. Do **batch-settlement channel deposits** work through the MCP wrapper (first paid call)?
4. Does the **EURe Permit2** path survive the MCP flow (the currency-variant cell)?
5. Does a paid `llm_complete` settle **usage-derived**, not the advertised ceiling (the chat's
   native authorize-ceiling/claim-usage split, inherited via the translation layer)?
6. (Deployed run) Does the **Scaleway gateway** pass POST/202/405 cleanly — flip `USE_DEPLOYED` below.

**Prerequisites:** the Deno kernel; `npm run dev:mcp` running; `TEST_WALLET_PRIVATE_KEY` in
`scw_js/.env` funded with Base Sepolia USDC, EURC and EURe (plus a little ETH for the channel
deposits' gas). Same wallet/channel state as the other buyer notebooks — see the channel-reuse
note below.

In [ ]:
// Setup: imports + env
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount } from "npm:viem@2/accounts";

// scw_js's own .env, one level up — the single source of truth (buyer key, receiver
// address, and the server's own secrets all live here; the server reads the same file).
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error(
        "TEST_WALLET_PRIVATE_KEY missing from scw_js/.env — add a key funded with Base Sepolia " +
        "USDC, EURC and EURe, plus a little ETH for the channel deposits' gas.",
    );
}
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

console.log("llm_mcp buyer notebook");
console.log(`   Buyer (payer): ${account.address}`);
console.log(`   Receiver     : ${env.NFT_WALLET_PUBLIC_KEY ?? "(not set — server will 500)"}`);

## Network selection

`llm_complete` offers Base Sepolia (the chat's own testnet offer — completions mock there),
so the default testnet run below pays with worthless coins and settles mocked completions.
A mainnet run is REAL MONEY: real Mistral billing per completion.

In [ ]:
import { base, baseSepolia, optimism } from "npm:viem@2/chains";

const USE_MAINNET = false;  // true = REAL MONEY: mainnet settlement + billed Mistral calls.
const USE_BASE = true;       // Optimism is mainnet-only for batch-settlement, see guard below.

if (!USE_BASE && !USE_MAINNET) {
    throw new Error(
        "Optimism Sepolia is unsupported for batch-settlement (no canonical contract). " +
        "Use Base Sepolia (USE_BASE = true) or an Optimism mainnet run (USE_MAINNET = true).",
    );
}

const NETWORK_CONFIG = {
    "base-testnet": {
        caip2Network: "eip155:84532" as const, networkName: "Base Sepolia (Testnet)",
        chain: baseSepolia, usdcName: "USDC",
        usdcAddress: "0x036CbD53842c5426634e7929541eC2318f3dCF7e" as `0x${string}`,
        eurcAddress: "0x808456652fdb597867f38412077A9182bf77359F" as `0x${string}` | undefined,
        // EURe (Monerium, 18 decimals, Permit2 + EIP-2612 permit — no approve(), no ETH for approval).
        eureAddress: "0x29F37F6adCa168B79B8d9567eab9BE3fBF21db85" as `0x${string}` | undefined,
        rpcUrl: undefined as string | undefined,
        explorer: "https://sepolia.basescan.org", faucet: "https://faucet.circle.com/",
    },
    "base-mainnet": {
        caip2Network: "eip155:8453" as const, networkName: "Base Mainnet",
        chain: base, usdcName: "USD Coin",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,
        eurcAddress: "0x60a3E35Cc302bFA44Cb288Bc5a4F316Fdb1adb42" as `0x${string}` | undefined,
        eureAddress: "0xbf6e2966A9C3D99C9E4D069E04f7Bdb9C8aa762C" as `0x${string}` | undefined,
        rpcUrl: "https://base-rpc.publicnode.com",
        explorer: "https://basescan.org", faucet: "Bridge: https://bridge.base.org",
    },
    "optimism-mainnet": {
        caip2Network: "eip155:10" as const, networkName: "Optimism Mainnet",
        chain: optimism, usdcName: "USD Coin",
        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85" as `0x${string}`,
        // No EURC on Optimism (Circle has no deployment there).
        eurcAddress: undefined as `0x${string}` | undefined,
        eureAddress: undefined as `0x${string}` | undefined,
        rpcUrl: "https://optimism-rpc.publicnode.com",
        explorer: "https://optimistic.etherscan.io", faucet: "Bridge: https://app.optimism.io/bridge",
    },
};
const configKey = USE_MAINNET ? "base-mainnet" : "base-testnet";
const config = NETWORK_CONFIG[configKey];
const NETWORK = config.caip2Network;

// Local: the dev server (`npm run dev:mcp`, scw_js/dev_server_mcp.ts on 127.0.0.1:8086).
// Deployed: the mcp function's custom domain (serverless.yml). Orthogonal to USE_MAINNET —
// the channel is on-chain state keyed by (buyer, receiver, network, asset), not by which
// server talked to it, so a channel opened here also pays the deployed endpoint, and vice
// versa (same property the search buyer notebook relies on to share the chat's channel).
const USE_DEPLOYED = false;
const MCP_URL = USE_DEPLOYED
    ? "https://mcp-agent.fretchen.eu/mcp"
    : "http://127.0.0.1:8086/mcp";

console.log(USE_MAINNET ? `REAL MONEY on ${config.networkName}` : `${config.networkName} (testnet)`);
console.log(`   ${NETWORK} • USDC ${config.usdcName} @ ${config.usdcAddress}`);
console.log(`   MCP endpoint: ${MCP_URL}`);

## Buyer setup — the channel scheme, same recipe as the REST buyers

The translation layer relays to `sc_llm_x402.handle`, which verifies and settles through the
same batch-settlement stack as the chat — so the client side is the recipe
`sc_llm_x402_buyer.ipynb` proved: `toClientEvmSigner` (with a `readContract`-capable public
client — the corrective-402 recovery path needs it), `ClientChannelStorage` backed by
`localStorage`, and a deposit floor of $0.50 so one channel covers the session.

In [ ]:
import { toClientEvmSigner } from "npm:@x402/evm@2.28.0";
import {
    BatchSettlementEvmScheme,
    type ClientChannelStorage,
    type BatchSettlementClientContext,
    type BatchSettlementDepositStrategyContext,
} from "npm:@x402/evm@2.28.0/batch-settlement/client";
import { createPublicClient, http } from "npm:viem@2";

const publicClient = createPublicClient({
    chain: config.chain,
    transport: http(config.rpcUrl),
});
const buyerSigner = toClientEvmSigner(
    { address: account.address, signTypedData: (a: unknown) => account.signTypedData(a as never) },
    publicClient,
);

class WebStorageClientChannelStorage implements ClientChannelStorage {
    constructor(private backend: Storage, private prefix = "x402-channel:") {}
    private keyFor(key: string) { return `${this.prefix}${key.toLowerCase()}`; }
    async get(key: string) {
        const raw = this.backend.getItem(this.keyFor(key));
        return raw ? (JSON.parse(raw) as BatchSettlementClientContext) : undefined;
    }
    async set(key: string, context: BatchSettlementClientContext) {
        this.backend.setItem(this.keyFor(key), JSON.stringify(context));
    }
    async delete(key: string) { this.backend.removeItem(this.keyFor(key)); }
}

// Same prefix as the other buyer notebooks: a channel here IS the chat's channel — a browser
// that has already chatted would pay llm_complete from the same voucher state.
const buyerStorage = new WebStorageClientChannelStorage(localStorage);

// Floor for channel deposits/top-ups, $0.50 in USDC atomic units — same value and same
// reasoning as useX402Chat.ts's MINIMUM_DEPOSIT_ATOMIC. EURe deposits are the SDK's own
// multiplier figure (its minimum exceeds the floor in EURe's 18-decimal units).
const MINIMUM_DEPOSIT_ATOMIC = 400_000n;

function depositStrategy(context: BatchSettlementDepositStrategyContext): string {
    const required = BigInt(context.minimumDepositAmount);
    return (required > MINIMUM_DEPOSIT_ATOMIC ? required : MINIMUM_DEPOSIT_ATOMIC).toString();
}

const buyerScheme = new BatchSettlementEvmScheme(buyerSigner, { storage: buyerStorage, depositStrategy });

// How many channel contexts exist right now — the channel-reuse assertions below count these.
function channelKeyCount(): number {
    let n = 0;
    for (let i = 0; i < localStorage.length; i++) {
        if (localStorage.key(i)!.startsWith("x402-channel:")) n++;
    }
    return n;
}
const channelsBefore = channelKeyCount();
console.log(`   Channel contexts in localStorage: ${channelsBefore}`);

## The x402 MCP client — two instances, two purposes

Both are the official `createx402MCPClient` — **no custom client code anywhere**:

- **observer** — `autoPayment: false`: an unpaid `tools/call` must throw the client's
  payment error (`code: 402`, demands on `err.paymentRequired`) — the convention check;
- **payer** — `autoPayment: true` with an `onPaymentRequested` logger: the wrapper must
  extract the demands, open/deposit the channel, pay in `_meta`, retry, and attach the
  settlement receipt (spike questions #2 and #3).

EURC and EURe are not in the SDK's default spend-control registry (USDC is), so both clients
allow-list them. The selector is pinned to USDC with the REAL signature
`(x402Version, filteredReqs)` — the x402 guide's one-arg example is out of date for
@x402/core 2.28.0.

In [ ]:
import { StreamableHTTPClientTransport } from "npm:@modelcontextprotocol/sdk@1.12.1/client/streamableHttp.js";
import { createx402MCPClient } from "npm:@x402/mcp@2.28.0";

function assert(cond: unknown, msg: string): asserts cond {
    if (!cond) throw new Error(`FAILED: ${msg}`);
    console.log(`ok: ${msg}`);
}

const allowedAssets = [
    ...(config.eurcAddress ? [{ network: NETWORK, asset: config.eurcAddress }] : []),
    ...(config.eureAddress ? [{ network: NETWORK, asset: config.eureAddress }] : []),
];

const logDemand = async (ctx: { toolName: string; paymentRequired: { accepts: unknown[] } }) => {
    console.log(`   payment demanded for ${ctx.toolName} (${ctx.paymentRequired.accepts.length} accepts):`);
    for (const a of ctx.paymentRequired.accepts as Array<{ network: string; asset: string; amount: string; payTo: string }>) {
        console.log(`     ${a.network} ${a.asset} amount=${a.amount} payTo=${a.payTo}`);
    }
    return true;
};

// Real selector signature (@x402/core 2.28.0): (x402Version, filteredReqs) — the reqs are
// ALREADY filtered to payable accepts; the return is used verbatim (no SDK fallback), so a
// miss must not return undefined: warn and fall back to the first payable accept.
function pinToAsset(asset: `0x${string}` | undefined) {
    return (_version: number, reqs: Array<{ asset?: string }>) => {
        const pinned = reqs.find((r) => (r.asset ?? "").toLowerCase() === (asset ?? "").toLowerCase());
        if (!pinned) {
            console.warn(`   pinned asset not payable; falling back to ${reqs[0]?.asset}`);
            return reqs[0];
        }
        return pinned;
    };
}

// connect() = initialize handshake. Spike question #1 lives here: our server is stateless
// JSON (405 on GET/DELETE); if the official transport chokes on that, it fails right here.
const observer = createx402MCPClient({
    name: "llm-mcp-buyer-notebook", version: "1.0.0",
    schemes: [{ network: NETWORK, client: buyerScheme }],
    spendControls: { allowedAssets },
    autoPayment: false,
});
await observer.connect(new StreamableHTTPClientTransport(new URL(MCP_URL)));
console.log("initialize handshake passed (stateless JSON accepted)");

const tools = await observer.listTools();
const toolNames = tools.tools.map((t: { name: string }) => t.name).sort();
console.log(`tools/list: ${JSON.stringify(toolNames)}`);
assert(toolNames.includes("llm_complete"), "server advertises llm_complete");
// The endpoint also serves web_search — mainnet-only by the handler's own rule; this
// notebook drives llm_complete only (the payable testnet walk).
assert(toolNames.includes("web_search"), "server advertises web_search too (not driven here)");

## Unpaid call — the convention check (spike question #2)

`autoPayment: false`, so nothing pays. The client's contract (verified from source): an
unpaid demand never comes back as a plain `callTool` return value — the wrapper extracts the
challenge from our payment-required result and **throws** (`code: 402`, demands on
`err.paymentRequired`). Reading the demands the clean way is the client's own probe,
`getToolPaymentRequirements`. NOTE: the package's `isPaymentRequiredError()` guard is NOT
applicable to the thrown error — it checks `error.data` (a JSON-RPC error shape), which this
client error does not carry.

In [ ]:
let thrown: unknown;
try {
    await observer.callTool("llm_complete", {
        model: "mistral-large-latest",
        messages: [{ role: "user", content: "hi" }],
    });
    throw new Error("expected the unpaid call to throw a payment-required error");
} catch (e) {
    thrown = e;
}

assert((thrown as { code?: unknown }).code === 402,
    "unpaid demand throws with code 402 (the client-side form of the payment convention)");
const thrownDemand = (thrown as { paymentRequired?: { accepts: Array<{ network: string }> } }).paymentRequired;
assert(Array.isArray(thrownDemand?.accepts) && thrownDemand.accepts.length > 0,
    "payment demands ride on err.paymentRequired through the official client");

const demand = await observer.getToolPaymentRequirements("llm_complete", {
    model: "mistral-large-latest",
    messages: [{ role: "user", content: "hi" }],
});
assert(demand !== null, "getToolPaymentRequirements returns the server's demands");
const networks = demand!.accepts.map((a: { network: string }) => a.network);
console.log(`   offered networks: ${JSON.stringify([...new Set(networks)])}`);
// The intentional asymmetry, served by the REAL chat handler: llm offers testnet
// (completions mock there), unlike search (mainnet-only).
assert(networks.includes(NETWORK), `the llm challenge offers this network (${NETWORK})`);

## Paid walk — first call opens the channel (spike questions #2 and #3)

The payer's first `llm_complete` must: read the demand, **deposit a real on-chain channel on
Base Sepolia** (a real transaction — the big unknown: do channel deposits work through the
MCP wrapper?), pay, retry with the payment in `params._meta["x402/payment"]`, and get the
settlement receipt back. On testnet the completion is the handler's own mock — no Mistral
spend. The second call must reuse the same channel — no new deposit.

In [ ]:
const payer = createx402MCPClient({
    name: "llm-mcp-buyer-notebook", version: "1.0.0",
    schemes: [{ network: NETWORK, client: buyerScheme }],
    spendControls: { allowedAssets },
    autoPayment: true,
    onPaymentRequested: logDemand,
    paymentRequirementsSelector: pinToAsset(config.usdcAddress),
});
await payer.connect(new StreamableHTTPClientTransport(new URL(MCP_URL)));

function printEvidence(label: string, r: { isError?: boolean; paymentMade?: boolean; content?: Array<{ text?: string }>; paymentResponse?: unknown }) {
    console.log(`${label} — isError: ${r.isError} | paymentMade: ${r.paymentMade}`);
    console.log(`   raw content[0]: ${String(r.content?.[0]?.text ?? "(empty)").slice(0, 300)}`);
    console.log(`   paymentResponse: ${JSON.stringify(r.paymentResponse)?.slice(0, 220)}`);
}

const t0 = performance.now();
const first = await payer.callTool("llm_complete", {
    model: "mistral-large-latest",
    messages: [{ role: "user", content: "In one sentence, what is an x402 payment channel?" }],
});
console.log(`llm_complete paid in ${Math.round(performance.now() - t0)}ms (first call: deposit + verify + settle)`);
printEvidence("first call", first);

const llm = JSON.parse(String(first.content?.[0]?.text ?? "{}"));
assert(llm.choices?.[0]?.message?.content, "paid llm_complete returns a completion");
const channelsAfterFirst = channelKeyCount();
console.log(`   channel contexts: ${channelsBefore} -> ${channelsAfterFirst}`);

// Second call: same channel, no deposit — the MCP wrapper must reuse the client-side
// channel context exactly like wrapFetchWithPayment does in the REST buyers.
const second = await payer.callTool("llm_complete", {
    model: "mistral-large-latest",
    messages: [{ role: "user", content: "And in one word?" }],
});
printEvidence("second call", second);
const channelsAfterSecond = channelKeyCount();
assert(channelsAfterSecond === channelsAfterFirst,
    `second call opened no new channel (${channelsAfterFirst} -> ${channelsAfterSecond})`);
const llm2 = JSON.parse(String(second.content?.[0]?.text ?? "{}"));
assert(llm2.choices?.[0]?.message?.content, "second paid llm_complete returns a completion");

## Usage-derived settlement (spike question #5)

The translation layer has no payment logic — the chat handler settles the real, usage-derived
amount natively (authorize the per-message ceiling, claim the actual tokens). The receipt is
the top-level `paymentResponse` field. Compare it against the completion's `usage` with the
Mistral rate card ($0.50/M input, $1.50/M output, 6 decimals for USDC — the same formula
`test/mcp_server.test.ts` pins). On testnet the completion is the mock and its usage is the
placeholder — a tiny claim IS the parity behavior: bill what it actually cost.

In [ ]:
const u = llm.usage as { prompt_tokens: number; completion_tokens: number } | undefined;
console.log(`   usage: ${JSON.stringify(llm.usage)}`);
if (u) {
    const expectedUsdc = (BigInt(u.prompt_tokens) * 50n + BigInt(u.completion_tokens) * 150n) / 100n;
    console.log(`   expected USDC claim for this usage: ${expectedUsdc} atomic ` +
        `(${expectedUsdc / 1_000_000n}.${(expectedUsdc % 1_000_000n).toString().padStart(6, "0")} USD)`);
    console.log("   -> compare against paymentResponse / the channel's claimed delta in the findings cell");
}
// A small usage that settled for the full advertised ceiling would be a parity bug —
// on testnet the mock's placeholder usage settles near-zero, which is the same rule.
assert(first.paymentResponse !== undefined, "the paid call carries a settlement receipt");

## Currency variants — pinning the accept explicitly (spike question #4)

The default walk paid USDC (the pinned selector). This cell pins EURC and EURe — each opens
its own channel (channels are per buyer/receiver/network/asset), and the EURe run is the
**Permit2 path** over the MCP flow: the permit rides the handler's challenge extensions, the
deposit needs no approval transaction.

In [ ]:
async function payWithAsset(label: string, asset: `0x${string}` | undefined) {
    if (!asset) { console.log(`   ${label}: not deployed on this network — skipped`); return; }
    const c = createx402MCPClient({
        name: "llm-mcp-buyer-notebook", version: "1.0.0",
        schemes: [{ network: NETWORK, client: buyerScheme }],
        spendControls: { allowedAssets },
        paymentRequirementsSelector: pinToAsset(asset),
    });
    await c.connect(new StreamableHTTPClientTransport(new URL(MCP_URL)));
    const r = await c.callTool("llm_complete", {
        model: "mistral-large-latest",
        messages: [{ role: "user", content: `A one-sentence reply paid in ${label}.` }],
    });
    printEvidence(label, r);
    const parsed = JSON.parse(String(r.content?.[0]?.text ?? "{}"));
    assert(parsed.choices?.[0]?.message?.content, `${label} variant paid and returned a completion`);
    await c.close();
}

await payWithAsset("EURC", config.eurcAddress);
await payWithAsset("EURe", config.eureAddress);
console.log(`channel contexts after variants: ${channelKeyCount()}`);

## Teardown

`close()` sends a `DELETE /mcp` — which a stateless server answers with 405. The SDK must
tolerate that silently (spike question #1, the DELETE half).

In [ ]:
await observer.close();
await payer.close();
console.log("closed both clients (DELETE answered 405 and the SDK tolerated it)");

## Findings

Fill in after a real run (testnet first, then flip `USE_DEPLOYED` for the gateway check):

- [ ] **Stateless server tolerated** — initialize + listTools + teardown passed against
      `http://127.0.0.1:8086/mcp` with no session, GET/DELETE answered 405
- [ ] **Payment convention verified** — the official client read the demands out of the
      unpaid result (code 402 + `err.paymentRequired`) and paid with zero custom code
- [ ] **Channel deposit over MCP works** — the first paid call deposited a real channel on Base Sepolia
- [ ] **EURe Permit2 over MCP works** — the EURe variant paid without an approval transaction
- [ ] **Usage-derived settlement** — the claim matched the completion's `usage` (near-zero for
      the testnet mock's placeholder usage), not the advertised ceiling
- [ ] **Gateway (deployed run)** — the same walk passed against `https://mcp-agent.fretchen.eu/mcp`

Channel reuse: `channelsBefore -> after first call -> after second -> after variants`:
`___ -> ___ -> ___ -> ___` (expected: one new channel per asset paid — USDC in the walk, EURC
and EURe in the variants — and zero on the second call).

`web_search` is deliberately not driven here: the handler's rule is mainnet-only, so a paid
search is a mainnet run (real Brave, $0.01/call).